In [1]:
! pip install "autoawq[eval]==0.2.9" "transformers==4.51.3" "datasets==2.21.0"

In [2]:
from awq.quantize.quantizer import AwqQuantizer
from awq import AutoAWQForCausalLM
from awq.evaluation import *
from transformers import AutoTokenizer
import os
import torch
from transformers.models.llama.modeling_llama import *
from awq.modules.linear import (
    WQLinear_GEMM,
    WQLinear_GEMV,
    WQLinear_Marlin,
    WQLinear_GEMVFast,
)
import importlib, model2bin
importlib.reload(model2bin)
from model2bin import *

# AutoAWQ 0.2.9 drops excluded linears before collecting calibration inputs.
# Keep them in calibration only; quantization/export behavior stays unchanged.
if not hasattr(AwqQuantizer, "_get_input_feat_unfiltered"):
    AwqQuantizer._get_input_feat_unfiltered = AwqQuantizer._get_input_feat
def _get_input_feat_with_excluded(self, layer, named_linears):
    calibration_linears = dict(named_linears)
    calibration_linears.setdefault("mlp.down_proj", layer.mlp.down_proj)
    return self._get_input_feat_unfiltered(layer, calibration_linears)
AwqQuantizer._get_input_feat = _get_input_feat_with_excluded

QUANT_GROUP = 128
SCALE_UP = 4

model_path = 'meta-llama/Llama-2-7b-chat-hf'
quant_path = '../meta-llama/Llama-2-7b-chat-hf-awq-fpga'
quant_config = { "zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMV" , "modules_to_not_convert": ["mlp.down_proj"]}

/home/changhong/anaconda3/envs/py310/lib/python3.10/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/home/changhong/anaconda3/envs/py310/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a

In [3]:
# quantize the model using AWQ

model = AutoAWQForCausalLM.from_pretrained(model_path, device_map="cuda:0")
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

model.quantize(tokenizer, quant_config=quant_config)

model.save_quantized(quant_path)
tokenizer.save_pretrained(quant_path)

Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.81s/it]
Repo card metadata block was not found. Setting CardData to empty.
2026-07-22:15:11:38,186 WARNING  [repocard.py:108] Repo card metadata block was not found. Setting CardData to empty.
AWQ: 100%|██████████| 32/32 [11:37<00:00, 21.79s/it]
2026-07-22:15:23:21,089 INFO     [_torch.py:293] The model is bigger than the maximum size per checkpoint (5GB). Model weighs have been saved in 2 checkpoint shards. You can find where each parameters has been saved in the index located at model.safetensors.index.json.
2026-07-22:15:23:21,089 INFO     [_torch.py:299] Model weights successfully saved to ../meta-llama/Llama-2-7b-chat-hf-awq-fpga!


('../meta-llama/Llama-2-7b-chat-hf-awq-fpga/tokenizer_config.json',
 '../meta-llama/Llama-2-7b-chat-hf-awq-fpga/special_tokens_map.json',
 '../meta-llama/Llama-2-7b-chat-hf-awq-fpga/tokenizer.json')

In [4]:
# directly load the quantized model

model = AutoAWQForCausalLM.from_quantized(quant_path, fuse_layers=False, device_map={"": 0}, max_memory={0: "18GiB"})
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

assert next(model.model.parameters()).device.type == "cuda"

Replacing layers...: 100%|██████████| 32/32 [00:03<00:00,  8.42it/s]


In [5]:
# gen_bin for kv260, the PS DDR address space is split into two parts

quant_model = LlamaModelFromAWQ(model.model, QUANT_GROUP, 1, [4, 1], SCALE_UP)

first_pack = quant_model.gen_model_bin_first_half(1024, 0)
print(len(first_pack))
second_pack = quant_model.gen_model_bin_second_half(1024, 0)
print(len(second_pack))

with open("llama0.bin", "wb") as f:
    f.write(first_pack)

with open("llama1.bin", "wb") as f:
    f.write(second_pack)

Embedding init done.
float16
NormFromAWQ init done.
LMHeadFromModel init done.
0
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
SplitDotFromAWQ init done.
float16
NormFromAWQ init done.
SplitSparseDotFromAWQ init done.
SplitDotFromAWQ init done.
SplitSparseAxpy init done.
--- Llama layer init done ---
1
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
SplitDotFromAWQ init done.
float16
NormFromAWQ init done.
SplitSparseDotFromAWQ init done.
SplitDotFromAWQ init done.
SplitSparseAxpy init done.
--- Llama layer init done ---
2
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
SplitDotFromAWQ init done.
float16
NormFromAWQ init done.
SplitSparseDotFromAWQ init done.
SplitDotFromAWQ init done.
SplitSparseAxpy init done.
--- Llama layer init done ---
3
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ

In [6]:
# gen_bin for zcu104 with PS and PL DDR

quant_model = LlamaModelFromAWQ(model.model, QUANT_GROUP, 2, [4, 1], SCALE_UP)

model_pack_ps = quant_model.gen_model_bin(1024, 0)
print(len(model_pack_ps))
model_pack_pl = quant_model.gen_model_bin(1024, 1)
print(len(model_pack_pl))

with open("llmps.bin", "wb") as f:
    f.write(model_pack_ps)

with open("llmpl.bin", "wb") as f:
    f.write(model_pack_pl)

Embedding init done.
float16
NormFromAWQ init done.
LMHeadFromModel init done.
0
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
SplitDotFromAWQ init done.
float16
NormFromAWQ init done.
SplitSparseDotFromAWQ init done.
SplitDotFromAWQ init done.
SplitSparseAxpy init done.
--- Llama layer init done ---
1
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
SplitDotFromAWQ init done.
float16
NormFromAWQ init done.
SplitSparseDotFromAWQ init done.
SplitDotFromAWQ init done.
SplitSparseAxpy init done.
--- Llama layer init done ---
2
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
SplitDotFromAWQ init done.
float16
NormFromAWQ init done.
SplitSparseDotFromAWQ init done.
SplitDotFromAWQ init done.
SplitSparseAxpy init done.
--- Llama layer init done ---
3
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ

In [7]:
# gen_bin for zcu104 with PL DDR only

quant_model = LlamaModelFromAWQ(model.model, QUANT_GROUP, 1, [1], SCALE_UP)

model_pack = quant_model.gen_model_bin(1024, 0)
print(len(model_pack))

with open("migpl.bin", "wb") as f:
    f.write(model_pack_ps)

Embedding init done.
float16
NormFromAWQ init done.
LMHeadFromModel init done.
0
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
SplitDotFromAWQ init done.
float16
NormFromAWQ init done.
SplitSparseDotFromAWQ init done.
SplitDotFromAWQ init done.
SplitSparseAxpy init done.
--- Llama layer init done ---
1
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
SplitDotFromAWQ init done.
float16
NormFromAWQ init done.
SplitSparseDotFromAWQ init done.
SplitDotFromAWQ init done.
SplitSparseAxpy init done.
--- Llama layer init done ---
2
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
SplitDotFromAWQ init done.
float16
NormFromAWQ init done.
SplitSparseDotFromAWQ init done.
SplitDotFromAWQ init done.
SplitSparseAxpy init done.
--- Llama layer init done ---
3
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ

In [8]:
# gen_bin for boards with two PL MIG DDRs

quant_model = LlamaModelFromAWQ(model.model, QUANT_GROUP, 2, [1, 1], SCALE_UP)

pack_0 = quant_model.gen_model_bin(1024, 0)
print(len(pack_0))
pack_1 = quant_model.gen_model_bin(1024, 1)
print(len(pack_1))

with open("llmc0.bin", "wb") as f:
    f.write(pack_0)

with open("llmc1.bin", "wb") as f:
    f.write(pack_1)

Embedding init done.
float16
NormFromAWQ init done.
LMHeadFromModel init done.
0
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
SplitDotFromAWQ init done.
float16
NormFromAWQ init done.
SplitSparseDotFromAWQ init done.
SplitDotFromAWQ init done.
SplitSparseAxpy init done.
--- Llama layer init done ---
1
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
SplitDotFromAWQ init done.
float16
NormFromAWQ init done.
SplitSparseDotFromAWQ init done.
SplitDotFromAWQ init done.
SplitSparseAxpy init done.
--- Llama layer init done ---
2
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
SplitDotFromAWQ init done.
float16
NormFromAWQ init done.
SplitSparseDotFromAWQ init done.
SplitDotFromAWQ init done.
SplitSparseAxpy init done.
--- Llama layer init done ---
3
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ

In [9]:
# gen_bin for boards with four PL MIG DDRs

quant_model = LlamaModelFromAWQ(model.model, QUANT_GROUP, 4, [1, 1, 1, 1], SCALE_UP)

pack_0 = quant_model.gen_model_bin(1024, 0)
print(len(pack_0))
pack_1 = quant_model.gen_model_bin(1024, 1)
print(len(pack_1))
pack_2 = quant_model.gen_model_bin(1024, 2)
print(len(pack_2))
pack_3 = quant_model.gen_model_bin(1024, 3)
print(len(pack_3))

with open("llmc0.bin", "wb") as f:
    f.write(pack_0)

with open("llmc1.bin", "wb") as f:
    f.write(pack_1)

with open("llmc2.bin", "wb") as f:
    f.write(pack_2)

with open("llmc3.bin", "wb") as f:
    f.write(pack_3)

Embedding init done.
float16
NormFromAWQ init done.
LMHeadFromModel init done.
0
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
SplitDotFromAWQ init done.
float16
NormFromAWQ init done.
SplitSparseDotFromAWQ init done.
SplitDotFromAWQ init done.
SplitSparseAxpy init done.
--- Llama layer init done ---
1
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
SplitDotFromAWQ init done.
float16
NormFromAWQ init done.
SplitSparseDotFromAWQ init done.
SplitDotFromAWQ init done.
SplitSparseAxpy init done.
--- Llama layer init done ---
2
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
AttnQKVFromAWQ init done.
SplitDotFromAWQ init done.
float16
NormFromAWQ init done.
SplitSparseDotFromAWQ init done.
SplitDotFromAWQ init done.
SplitSparseAxpy init done.
--- Llama layer init done ---
3
float16
AttnNormFromAWQ init done.
AttnQKVFromAWQ

In [ ]:
# SHA-256 values for the generated bin files
!sha256sum llama0.bin llama1.bin llmps.bin llmpl.bin migpl.bin llmc0.bin llmc1.bin llmc2.bin llmc3.bin

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


45bb125d50787badcc6df85fd99ce499ea3a43e160dcee4d736f6fa1b5c2c093  llama0.bin
7e947c152ef71de1128248c25a5bda18c652e9356a3ed00c0953ea17ad294afe  llama1.bin
8e76c8e6f2d12de8991693eec8ac9353e37f23e1d174e55c0f84f5f91bf69a78  llmps.bin
eb97cc8a11a7d2824b388815204cab8a9e865783fd1aaa9ee41aaf2e32fe81ce  llmpl.bin
8e76c8e6f2d12de8991693eec8ac9353e37f23e1d174e55c0f84f5f91bf69a78  migpl.bin
3f4da5a2b94f4de9b4cb00e14da57a564777b5625b68aebe290863e7e2ea60b5  llmc0.bin
2e3d7feac3d1ef4bb77372efc13fd12c688ef7c8547c5801aa10b722ac88eda4  llmc1.bin
35963be2d739bc783c020f2408591a876377cd2d69733462d6c8ab9127ed4bbc  llmc2.bin
1931c58890a504b135dccf8e476e666b8d1a8cb887a03b2db36b419f5166de11  llmc3.bin
